# Arena Data Alignment Test and Verification

This notebook tests the arena data alignment to Open Ephys timebase and provides an interactive viewer for screen touch events with LFP traces.

## Overview

1. Load block and parse Open Ephys events
2. Load aligned arena CSVs (bug_trajectory, app_events, screen_touches, trials_data)
3. Verify alignment by comparing time ranges
4. Interactive Bokeh viewer: screen touches + LFP traces (similar to LED blink verification)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from eye_tracking_system_tools.preprocessing import BlockSync, load_aligned_arena_data
from bokeh.io import output_notebook, show, reset_output
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, CustomJS, Span, Div, Button
from bokeh.layouts import column as bokeh_column, row as bokeh_row

output_notebook()

Loading BokehJS ...

In [2]:
# Configuration
animal = "PV_208"
date = "2025_12_14"
block_num = "019"

base_path = Path(r"D:\sample_data_for_eye_repo")
block_path = base_path / animal / date / f"block_{block_num}"
arena_videos_dir = block_path / "arena_videos"

print(f"Block path: {block_path}")
print(f"Arena videos dir: {arena_videos_dir}")

Block path: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019
Arena videos dir: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019\arena_videos


In [6]:
# Load block and parse Open Ephys events
block = BlockSync(animal,date,block_num,base_path)

# Set channel mapping for PV_208
block.channeldict = {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"}

# Parse events
block.parse_open_ephys_events()

print(f"Sample rate: {block.sample_rate} Hz")
print(f"OE events shape: {block.oe_events.shape}")
print(f"First Arena_TTL sample: {block.oe_events['Arena_TTL'].dropna().iloc[0]}")

instantiated block number 019 at Path: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019, new OE version
Found the sample rate for block 019 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_208\2025_12_14\block_019\oe_files\PV208_d5t2_2025-12-14_12-29-53\Record Node 106...

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode)
retrieving zertoh sample number for block 019
got it!
running parse_open_ephys_events...
block 019 has a parsed events file, reading...
Sample rate: 20000 Hz
OE events shape: (335544, 9)
First Arena_TTL sample: 308.0


In [7]:
# Load aligned arena data
arena_data = load_aligned_arena_data(block, arena_videos_dir)

print("\nLoaded arena CSVs:")
for name, df in arena_data.items():
    print(f"  {name}: {df.shape[0]} rows, columns: {list(df.columns)}")
    if 'ms_axis' in df.columns:
        print(f"    ms_axis range: {df['ms_axis'].min():.1f} - {df['ms_axis'].max():.1f} ms")
    if 'ms_axis_start' in df.columns:
        print(f"    ms_axis_start range: {df['ms_axis_start'].min():.1f} - {df['ms_axis_start'].max():.1f} ms")
        print(f"    ms_axis_end range: {df['ms_axis_end'].min():.1f} - {df['ms_axis_end'].max():.1f} ms")


Loaded arena CSVs:
  bug_trajectory: 65901 rows, columns: ['Unnamed: 0', 'time', 'x', 'y', 'ms_axis']
    ms_axis range: 10315.2 - 1810457.2 ms
  app_events: 25 rows, columns: ['Unnamed: 0', 'time', 'event', 'ms_axis']
    ms_axis range: 441425.2 - 1749558.2 ms
  screen_touches: 8 rows, columns: ['Unnamed: 0', 'time', 'x', 'y', 'bug_x', 'bug_y', 'is_hit', 'is_reward_any_touch', 'is_reward_bug', 'is_climbing', 'bug_type', 'bug_size', 'in_block_trial_id', 'trial_id', 'ms_axis']
    ms_axis range: 148157.2 - 1801092.2 ms
  trials_data: 15 rows, columns: ['Unnamed: 0', 'trial_db_id', 'start_time', 'trial_bugs', 'bug_sizes', 'bug_speed', 'exit_hole', 'extra', 'duration', 'end_time', 'ms_axis_start', 'ms_axis_end']
    ms_axis_start range: 10301.2 - 1728266.2 ms
    ms_axis_end range: 92160.2 - 1810458.2 ms


## Alignment Verification

Compare arena data time ranges with block's final_sync_df ms_axis to verify alignment.

In [8]:
# Load final_sync_df if available
if hasattr(block, 'final_sync_df') and block.final_sync_df is not None:
    sync_df = block.final_sync_df
else:
    # Try loading from disk
    from eye_tracking_system_tools.preprocessing.block_sync_core import load_final_sync_df
    try:
        sync_df = load_final_sync_df(block, verbose=True)
    except FileNotFoundError:
        print("final_sync_df not available. Skipping verification.")
        sync_df = None

if sync_df is not None and 'ms_axis' in sync_df.columns:
    sync_ms_min = sync_df['ms_axis'].min()
    sync_ms_max = sync_df['ms_axis'].max()
    print(f"\nFinal sync df ms_axis range: {sync_ms_min:.1f} - {sync_ms_max:.1f} ms")
    
    # Compare with arena data
    if 'bug_trajectory' in arena_data:
        bug_ms_min = arena_data['bug_trajectory']['ms_axis'].min()
        bug_ms_max = arena_data['bug_trajectory']['ms_axis'].max()
        print(f"Bug trajectory ms_axis range: {bug_ms_min:.1f} - {bug_ms_max:.1f} ms")
        print(f"  Overlap: {max(sync_ms_min, bug_ms_min):.1f} - {min(sync_ms_max, bug_ms_max):.1f} ms")
    
    if 'trials_data' in arena_data:
        trials_ms_min = arena_data['trials_data']['ms_axis_start'].min()
        trials_ms_max = arena_data['trials_data']['ms_axis_end'].max()
        print(f"Trials ms_axis range: {trials_ms_min:.1f} - {trials_ms_max:.1f} ms")
        print(f"  Overlap: {max(sync_ms_min, trials_ms_min):.1f} - {min(sync_ms_max, trials_ms_max):.1f} ms")

[INFO] Found multiple sync files: ['final_sync_df.csv', 'blocksync_df.csv']. Using newest: final_sync_df.csv
[OK] Loaded final_sync_df.csv → block.final_sync_df (rows=113,056)


## Screen Touch + LFP Viewer

Interactive Bokeh viewer showing screen touch events with their associated LFP perturbations. Similar to LED blink verification.

In [ ]:
def plot_screen_touch_events_viewer(
    block,
    screen_touches_df: pd.DataFrame,
    channel: int = 1,
    window_half_s: float = 0.25,
    to_browser: bool = True
):
    """
    Interactive viewer: windows around each screen touch event; Prev/Next to step.
    Full-resolution LFP + touch event info. Requires block.oe_rec.
    """
    if not hasattr(block, "oe_rec") or block.oe_rec is None:
        raise ValueError("block.oe_rec is not available.")
    
    if 'ms_axis' not in screen_touches_df.columns:
        raise ValueError("screen_touches_df must have 'ms_axis' column.")
    
    fs = block.sample_rate if hasattr(block, "sample_rate") else block.get_sample_rate()
    
    # Get touch events with valid ms_axis
    touches = screen_touches_df[screen_touches_df['ms_axis'].notna()].copy()
    if len(touches) == 0:
        raise ValueError("No valid touch events found.")
    
    # Convert ms_axis to seconds (OE timebase)
    # ms_axis is ms from OE recording start, so divide by 1000
    touch_times_s = touches['ms_axis'].values / 1000.0
    n_events = len(touch_times_s)
    
    window_ms = 2 * window_half_s * 1000
    print(f"Extracting {n_events} windows of {window_ms:.0f} ms around screen touch events...")
    
    all_lfp_t = []
    all_lfp_y = []
    event_info = []
    
    for i, t_center_s in enumerate(touch_times_s):
        # Convert ms_axis (ms from recording start) back to OE sample number
        # ms_axis = Arena_TTL / (sample_rate/1000), so Arena_TTL = ms_axis * (sample_rate/1000)
        ms_axis_val = touches.iloc[i]['ms_axis']
        arena_ttl_sample = int(ms_axis_val * (fs / 1000.0))
        
        # Convert sample to seconds (like LED viewer does)
        t_center_s_from_sample = arena_ttl_sample / fs
        start_ms = (t_center_s_from_sample - window_half_s) * 1000
        
        start_time_ms = np.array([[start_ms]])
        
        try:
            chunk_data, chunk_timestamps = block.oe_rec.get_data(
                channels=[channel],
                start_time_ms=start_time_ms,
                window_ms=window_ms,
                convert_microvolts=True,
                return_timestamps=True,
                repress_output=True,
            )
        except Exception as e:
            print(f"  Event {i+1}: get_data failed ({e}), skipping")
            continue
        
        if chunk_data is None or chunk_timestamps is None:
            continue
        
        # Convert timestamps to seconds relative to window start
        lfp_timestamps_ms = chunk_timestamps[0, :]
        lfp_t = (lfp_timestamps_ms - lfp_timestamps_ms[0]) / 1000.0  # Relative to window start
        lfp_y = chunk_data[0, 0, :]
        
        all_lfp_t.append(lfp_t.tolist())
        all_lfp_y.append(lfp_y.tolist())
        
        # Store event info
        row = touches.iloc[i]
        info = {
            'time_s': t_center_s,
            'x': row.get('x', np.nan),
            'y': row.get('y', np.nan),
            'bug_x': row.get('bug_x', np.nan),
            'bug_y': row.get('bug_y', np.nan),
            'is_hit': row.get('is_hit', False),
            'is_reward_any_touch': row.get('is_reward_any_touch', False),
            'is_reward_bug': row.get('is_reward_bug', False),
            'bug_type': row.get('bug_type', 'unknown'),
            'bug_size': row.get('bug_size', np.nan),
        }
        event_info.append(info)
    
    n_events = len(all_lfp_t)
    if n_events == 0:
        raise ValueError("Failed to extract any event windows.")
    
    # Create Bokeh data sources
    cds_ix = ColumnDataSource(dict(ix=[0]))
    cds_lfp = ColumnDataSource(dict(x=all_lfp_t[0], y=all_lfp_y[0]))
    
    # Create figure
    p_lfp = figure(
        width=1000, height=400,
        title="LFP around screen touch event",
        x_axis_label="Time relative to touch (s)", y_axis_label="Voltage (µV)",
        tools="pan,box_zoom,wheel_zoom,reset,save",
    )
    p_lfp.line("x", "y", source=cds_lfp, line_width=1.5, color="black")
    
    # Vertical line at touch time (center of window)
    span_touch = Span(
        location=window_half_s, dimension="height",
        line_color="#d62728", line_dash="dashed", line_width=2
    )
    p_lfp.add_layout(span_touch)
    
    # Info div
    info_0 = event_info[0]
    info_text = (
        f"<b>Touch Event 1 / {n_events}</b><br>"
        f"Time: {info_0['time_s']:.3f} s<br>"
        f"Touch position: ({info_0['x']:.0f}, {info_0['y']:.0f})<br>"
        f"Bug position: ({info_0['bug_x']:.2f}, {info_0['bug_y']:.2f})<br>"
        f"Hit: {info_0['is_hit']}<br>"
        f"Reward (any touch): {info_0['is_reward_any_touch']}<br>"
        f"Reward (bug): {info_0['is_reward_bug']}<br>"
        f"Bug type: {info_0['bug_type']}, size: {info_0['bug_size']:.0f}"
    )
    info_div = Div(text=info_text, width=400, height=200, styles={"font-size": "12px"})
    
    # Navigation callbacks
    def make_callback(delta):
        return CustomJS(
            args=dict(
                cds_ix=cds_ix, cds_lfp=cds_lfp,
                all_lfp_t=all_lfp_t, all_lfp_y=all_lfp_y,
                n_events=n_events, info_div=info_div,
                event_info=event_info, window_half_s=window_half_s,
            ),
            code="""
                var cur = cds_ix.data.ix[0];
                cur = (cur + """ + str(delta) + """ + n_events) % n_events;
                cds_ix.data = { ix: [cur] };
                cds_lfp.data = { x: all_lfp_t[cur], y: all_lfp_y[cur] };
                
                var info = event_info[cur];
                info_div.text = "<b>Touch Event " + (cur+1) + " / " + n_events + "</b><br>" +
                    "Time: " + info.time_s.toFixed(3) + " s<br>" +
                    "Touch position: (" + info.x.toFixed(0) + ", " + info.y.toFixed(0) + ")<br>" +
                    "Bug position: (" + info.bug_x.toFixed(2) + ", " + info.bug_y.toFixed(2) + ")<br>" +
                    "Hit: " + info.is_hit + "<br>" +
                    "Reward (any touch): " + info.is_reward_any_touch + "<br>" +
                    "Reward (bug): " + info.is_reward_bug + "<br>" +
                    "Bug type: " + info.bug_type + ", size: " + info.bug_size.toFixed(0);
            """,
        )
    
    btn_prev = Button(label="Previous event")
    btn_prev.js_on_click(make_callback(-1))
    btn_next = Button(label="Next event")
    btn_next.js_on_click(make_callback(1))
    
    reset_output()
    layout = bokeh_column(
        bokeh_row(btn_prev, btn_next),
        bokeh_row(p_lfp, info_div),
    )
    
    if to_browser:
        show(layout)
        print(f"Viewer ready: {n_events} events, full-resolution LFP in ±{window_half_s}s around each touch.")
    else:
        return layout

In [ ]:
# Run the viewer
if 'screen_touches' in arena_data:
    plot_screen_touch_events_viewer(
        block,
        arena_data['screen_touches'],
        channel=1,  # LFP channel
        window_half_s=0.25,  # ±250 ms around each touch
        to_browser=True
    )
else:
    print("screen_touches not found in arena_data")